In [3]:
'''
Source data for the tables of the paper

This script creates the source data for all the tables of the paper. It saves TSV
files with the main statistics for the age effects on the aperiodic parameters, 
including the beta estimate, SD, T value, and uncorrected and corrected p-values 
for all the models, for each sensor type.

Date: May-2026 (created)

'''


import docx
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os
import pandas as pd
import numpy as np
import sys

if os.name == 'nt':
    cfgdir = r"U:\Documents\CamCAN\code\maipy"
else:
    cfgdir = "/imaging/camcan/sandbox/mc06/code/maipy"

sys.path.insert(1, cfgdir)
import mcgdirs as dirs

statsdir = os.path.join(dirs.mysandboxdatadir, 'BIDS_long_p5_rest_arm1','derivatives','aperiodic_filt0.1-145Hz_fs300Hz_trans_z44mm','stats')

statsdir_notrans = os.path.join(dirs.mysandboxdatadir, 'BIDS_long_p5_rest_arm1','derivatives','aperiodic_filt0.1-145Hz_fs300Hz','stats')

analyses_dict = {
    'basic_main': {'stat_folder': 'lme_maxT_finley_2betas_10000rand', 'trans': True},
    'onecov_emptyroom': {'stat_folder': 'lme_1cov_emptyroomtotalminusaperiodicrest_maxT_finley_2betas_ecg04eog08_10000rand', 'trans': True},
    'onecov_ecg': {'stat_folder': 'lme_1cov_ecglikemegtotalrelpow2-40Hz_maxT_finley_2betas_ecg04eog08_10000rand', 'trans': True},
    'sixcov': {'stat_folder': 'lme_allcov_maxT_finley_2betas_ecg04eog08_10000rand', 'trans': True},
    'control_ignorepeaks': {'stat_folder': 'lme_maxT_finley_2betas_nonoisepeaks_ecg04eog08_10000rand', 'trans': True},
    'control_nointerpol': {'stat_folder': 'lme_maxT_finley_2betas_nointerp_ecg04eog08_10000rand', 'trans': True},
    'control_cardiacMEGonly': {'stat_folder': 'lme_maxT_finley_2betas_allbutecg04_10000rand', 'trans': True},
    'control_withcardiac': {'stat_folder': 'lme_maxT_finley_2betas_eog08_10000rand', 'trans': True},
    'control_ECGchannel': {'stat_folder': 'lme_maxT_finley_2betas_ECGlikemeg_10000rand', 'trans': False},
    'control_emptyroom': {'stat_folder': 'lme_maxT_finley_2betas_filt_emptyroom_10000rand', 'trans': False},
}

megtype = 'grad'

for analysis_name, analysis_info in analyses_dict.items():

    # Check if the stats directory for the current analysis exists
    if analysis_info['trans']:
        stats_dir = os.path.join(statsdir, analysis_info['stat_folder'])
    else:
        stats_dir = os.path.join(statsdir_notrans, analysis_info['stat_folder'])
        
    if not os.path.isdir(stats_dir):
            raise ValueError(f"Stats directory {stats_dir} for analysis {analysis_name} not found.")
    
    analysis_info['stats_dir'] = stats_dir
    
    files = os.listdir(stats_dir)

    # Find the corrected p-value file for the current analysis
    if analysis_name == 'control_ECGchannel':
        correctedfiles = [f for f in files if f'maxT_corrected_pvals' in f and f.endswith('.npy')]
    else:
        correctedfiles = [f for f in files if f'maxT{megtype}_corrected_pvals' in f and f.endswith('.npy')]

    if len(correctedfiles) > 1:
        print(correctedfiles)
        raise ValueError(f'Multiple corrected p-value files found for analysis {analysis_name}')
    elif len(correctedfiles) == 0:
        raise ValueError(f'No corrected p-value file found for analysis {analysis_name}')
    
    print(f"Analysis {analysis_name}: Found corrected p-value file: {correctedfiles[0]}")

    analysis_info['corrected_pvals_path'] = os.path.join(stats_dir, correctedfiles[0])

    # Find the stat_ori file for the current analysis
    statorifiles = [f for f in files if f.endswith('_statori.npy')]
    if len(statorifiles) > 1:
        print(statorifiles)
        raise ValueError(f'Multiple stat_ori files found for analysis {analysis_name}')
    elif len(statorifiles) == 0:
        raise ValueError(f'No stat_ori file found for analysis {analysis_name}')
    
    print(f"Analysis {analysis_name}: Found stat_ori file: {statorifiles[0]}")

    analysis_info['statori_path'] = os.path.join(stats_dir, statorifiles[0])

# ---- end of loop to check files and add paths to analyses_dict ----

# Loop over spectral parameters of interest and create dataframes with the main statistics

parameters = ['exponent']
bands = ['theta', 'alpha', 'beta', 'gamma', 'low_alpha', 'high_alpha', 'low_beta', 'high_beta']

for band in bands:
    for measure in ['band_power', 'peak_freq']:
        parameters.append(f'{band}_{measure}')



df_list = []

for analysis_name, analysis_info in analyses_dict.items():
    stats_dir = analysis_info['stats_dir']

    corrected_pvals_path = analysis_info['corrected_pvals_path']
    statori_path = analysis_info['statori_path']

    # Load corrected p-values and statori
    corrected_pvals = np.load(corrected_pvals_path)
    statori = np.load(statori_path, allow_pickle=True).item()

    vars = statori['vars']

    if analysis_name != 'control_ECGchannel':            
        vars = [v for v in vars if megtype in v]

    df_vars_list = []

    for var in parameters:
        print(var)
        # Read corrected p-values and statori for the current analysis

        var_name = [v for v in vars if var in v]

        if len(var_name) == 0:
            print(f'Variable {var} not found in statori for analysis {analysis_name}. Skipping.')
            continue
        
        else:
            var_name = var_name[0]
            var_idx = vars.index(var_name)
            resultsfile = f'{var_name}_lme_results.tsv'
            print(var_name)

        pval_corrected = corrected_pvals[var_idx, :]

        if os.path.isfile(os.path.join(stats_dir, resultsfile)):
            results_df = pd.read_csv(os.path.join(stats_dir, resultsfile), sep='\t').set_index('Unnamed: 0')

            df_tmp = results_df.copy()
            df_tmp = df_tmp[['Estimate', 'SE', 'T-stat', 'DF', 'P-val']]
            df_tmp = df_tmp.loc[['Age0', 'deltaAge', 'Age0:deltaAge']]
            df_tmp['P-corrected'] = pval_corrected
            df_tmp['Parameter'] = var

            df_tmp = df_tmp.reset_index()
            df_tmp = df_tmp.rename(columns={'Unnamed: 0': 'Effect'})
            df_tmp = df_tmp[['Parameter', 'Effect', 'Estimate', 'SE', 'T-stat', 'DF', 'P-val', 'P-corrected']]
            df_tmp = df_tmp.set_index(['Parameter', 'Effect'])

            df_tmp.rename(columns={'Estimate': f'{analysis_name}_Estimate', 'SE': f'{analysis_name}_SE', 'T-stat': f'{analysis_name}_T-stat', 'DF': f'{analysis_name}_DF', 'P-val': f'{analysis_name}_P-val', 'P-corrected': f'{analysis_name}_P-corrected'}, inplace=True)

            df_vars_list.append(df_tmp)
        
        else:
            print(f'Results file {resultsfile} not found for variable {var_name} in analysis {analysis_name}. Skipping.')
            continue

    df_vars = pd.concat(df_vars_list, axis=0)

    df_list.append(df_vars)

df_output = pd.concat(df_list, axis=1)
df_output.to_csv(f'tables_source_data_{megtype}.tsv', sep='\t')

# --- Repeat the process for magnetometers ---

megtype = 'mag'

for analysis_name, analysis_info in analyses_dict.items():

    # Check if the stats directory for the current analysis exists
    if analysis_info['trans']:
        stats_dir = os.path.join(statsdir, analysis_info['stat_folder'])
    else:
        stats_dir = os.path.join(statsdir_notrans, analysis_info['stat_folder'])
        
    if not os.path.isdir(stats_dir):
            raise ValueError(f"Stats directory {stats_dir} for analysis {analysis_name} not found.")
    
    analysis_info['stats_dir'] = stats_dir
    
    files = os.listdir(stats_dir)

    # Find the corrected p-value file for the current analysis
    if analysis_name == 'control_ECGchannel':
        correctedfiles = [f for f in files if f'maxT_corrected_pvals' in f and f.endswith('.npy')]
    else:
        correctedfiles = [f for f in files if f'maxT{megtype}_corrected_pvals' in f and f.endswith('.npy')]

    if len(correctedfiles) > 1:
        print(correctedfiles)
        raise ValueError(f'Multiple corrected p-value files found for analysis {analysis_name}')
    elif len(correctedfiles) == 0:
        raise ValueError(f'No corrected p-value file found for analysis {analysis_name}')
    
    print(f"Analysis {analysis_name}: Found corrected p-value file: {correctedfiles[0]}")

    analysis_info['corrected_pvals_path'] = os.path.join(stats_dir, correctedfiles[0])

    # Find the stat_ori file for the current analysis
    statorifiles = [f for f in files if f.endswith('_statori.npy')]
    if len(statorifiles) > 1:
        print(statorifiles)
        raise ValueError(f'Multiple stat_ori files found for analysis {analysis_name}')
    elif len(statorifiles) == 0:
        raise ValueError(f'No stat_ori file found for analysis {analysis_name}')
    
    print(f"Analysis {analysis_name}: Found stat_ori file: {statorifiles[0]}")

    analysis_info['statori_path'] = os.path.join(stats_dir, statorifiles[0])

# ---- end of loop to check files and add paths to analyses_dict ----

# Loop over spectral parameters of interest and create dataframes with the main statistics

parameters = ['exponent']
bands = ['theta', 'alpha', 'beta', 'gamma', 'low_alpha', 'high_alpha', 'low_beta', 'high_beta']

for band in bands:
    for measure in ['band_power', 'peak_freq']:
        parameters.append(f'{band}_{measure}')



df_list = []

for analysis_name, analysis_info in analyses_dict.items():
    stats_dir = analysis_info['stats_dir']

    corrected_pvals_path = analysis_info['corrected_pvals_path']
    statori_path = analysis_info['statori_path']

    # Load corrected p-values and statori
    corrected_pvals = np.load(corrected_pvals_path)
    statori = np.load(statori_path, allow_pickle=True).item()

    vars = statori['vars']

    if analysis_name != 'control_ECGchannel':            
        vars = [v for v in vars if megtype in v]

    df_vars_list = []

    for var in parameters:
        print(var)
        # Read corrected p-values and statori for the current analysis

        var_name = [v for v in vars if var in v]

        if len(var_name) == 0:
            print(f'Variable {var} not found in statori for analysis {analysis_name}. Skipping.')
            continue
        
        else:
            var_name = var_name[0]
            var_idx = vars.index(var_name)
            resultsfile = f'{var_name}_lme_results.tsv'
            print(var_name)

        pval_corrected = corrected_pvals[var_idx, :]

        if os.path.isfile(os.path.join(stats_dir, resultsfile)):
            results_df = pd.read_csv(os.path.join(stats_dir, resultsfile), sep='\t').set_index('Unnamed: 0')

            df_tmp = results_df.copy()
            df_tmp = df_tmp[['Estimate', 'SE', 'T-stat', 'DF', 'P-val']]
            df_tmp = df_tmp.loc[['Age0', 'deltaAge', 'Age0:deltaAge']]
            df_tmp['P-corrected'] = pval_corrected
            df_tmp['Parameter'] = var

            df_tmp = df_tmp.reset_index()
            df_tmp = df_tmp.rename(columns={'Unnamed: 0': 'Effect'})
            df_tmp = df_tmp[['Parameter', 'Effect', 'Estimate', 'SE', 'T-stat', 'DF', 'P-val', 'P-corrected']]
            df_tmp = df_tmp.set_index(['Parameter', 'Effect'])

            df_tmp.rename(columns={'Estimate': f'{analysis_name}_Estimate', 'SE': f'{analysis_name}_SE', 'T-stat': f'{analysis_name}_T-stat', 'DF': f'{analysis_name}_DF', 'P-val': f'{analysis_name}_P-val', 'P-corrected': f'{analysis_name}_P-corrected'}, inplace=True)

            df_vars_list.append(df_tmp)
        
        else:
            print(f'Results file {resultsfile} not found for variable {var_name} in analysis {analysis_name}. Skipping.')
            continue

    df_vars = pd.concat(df_vars_list, axis=0)

    df_list.append(df_vars)

df_output = pd.concat(df_list, axis=1)
df_output.to_csv(f'tables_source_data_{megtype}.tsv', sep='\t')

Analysis basic_main: Found corrected p-value file: aperiodic_stier_filtecg04eog08_finley_maxTgrad_corrected_pvals.npy
Analysis basic_main: Found stat_ori file: aperiodic_stier_filtecg04eog08_finley_lme_statori.npy
Analysis onecov_emptyroom: Found corrected p-value file: aperiodic_stier_filtecg04eog08_finley_maxTgrad_corrected_pvals.npy
Analysis onecov_emptyroom: Found stat_ori file: aperiodic_stier_filtecg04eog08_finley_lme_1cov_statori.npy
Analysis onecov_ecg: Found corrected p-value file: aperiodic_stier_filtecg04eog08_finley_maxTgrad_corrected_pvals.npy
Analysis onecov_ecg: Found stat_ori file: aperiodic_stier_filtecg04eog08_finley_lme_1cov_statori.npy
Analysis sixcov: Found corrected p-value file: aperiodic_stier_filtecg04eog08_finley_maxTgrad_corrected_pvals.npy
Analysis sixcov: Found stat_ori file: aperiodic_stier_filtecg04eog08_finley_lme_allcov_statori.npy
Analysis control_ignorepeaks: Found corrected p-value file: aperiodic_stier_filtecg04eog08_finley_maxTgrad_corrected_pvals.